# 🛫 SkyAnalyst-AI: Semantic Analysis & RAG Prototyping

**Researcher:** Sinem Demirci

**Objective:** Exploratory Data Analysis (EDA) of cabin reports and prototyping the Vector Database (RAG) architecture.

---

### 1. Environment Setup & Data Loading

*Loading the synthetic aviation dataset and initializing core libraries.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Load dataset
df = pd.read_csv('../data/cabin_reports.csv')
print(f"Dataset loaded with {df.shape[0]} reports.")
df.head()

In [ ]:
from textblob import TextBlob
from sentence_transformers import SentenceTransformer
import chromadb

### 2. Exploratory Data Analysis (EDA)

*Understanding operational patterns and incident distributions.*

In [ ]:
# Distribution of report categories
print("--- Incident Distribution by Category ---")
print(df['category'].value_counts(normalize=True) * 100)

# Statistical Analysis: Flight Duration vs Category
# Insight: Identifying if long-haul flights correlate with specific incident types
plt.figure(figsize=(12, 6))
sns.boxplot(x='category', y='duration_min', data=df, hue='category', palette="viridis", legend=False)
plt.title('Flight Duration Distribution by Incident Category')
plt.ylabel('Duration (Minutes)')
plt.xlabel('Incident Category')
plt.show()

**Business Insight:** Long-haul flights (>500 mins) show a higher density of **Medical** and **Passenger** related reports, suggesting a need for specialized crew training on these routes.

---

### 3. Sentiment Analysis Layer

*Quantifying the emotional intensity of cabin narratives.*


In [ ]:
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df['sentiment_score'] = df['report_content'].apply(get_sentiment)

# Visualizing Sentiment Polarity
plt.figure(figsize=(10, 6))
sns.violinplot(x='category', y='sentiment_score', data=df, inner="quart", palette="magma")
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.title('Sentiment Intensity Distribution per Category')
plt.show()

**Research Note:** **Medical** incidents consistently exhibit the highest negative polarity, serving as a primary trigger for urgent operational reviews.

---

### 4. Embedding Prototype (Sentence Transformers)

*Transforming unstructured text into high-dimensional semantic vectors.*

In [ ]:
# Initializing the MiniLM-L6-v2 model for balanced speed and accuracy
model = SentenceTransformer('all-MiniLM-L6-v2')

# Prototyping an embedding
sample_report = df['report_content'].iloc[0]
vector_sample = model.encode(sample_report)

print(f"Original Text: {sample_report[:100]}...")
print(f"Vector Dimensions: {len(vector_sample)}")

### 5. Vector Database Integration (ChromaDB)

*Establishing a persistent RAG layer for semantic retrieval.*

In [ ]:
# Initialize persistent client
client = chromadb.PersistentClient(path="../data/cabin_vector_db")

# Create/Retrieve Collection
collection = client.get_or_create_collection(name="cabin_reports")

# PREPARATION: Ensure unique IDs within the input list
# Using row index (ID-0, ID-1...) is the safest way for synthetic data
ids = [f"ID-{i}" for i in range(len(df))]
documents = df['report_content'].tolist()
embeddings = model.encode(documents).tolist()
metadatas = df[['category', 'route', 'urgency']].to_dict(orient='records')

print("Indexing reports into ChromaDB using upsert...")
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Indexing complete. Total records in Vector DB: {collection.count()}")

### 6. Semantic Query Testing

*Validating the RAG engine with operational queries.*

In [ ]:
query = "Issues with kitchen ovens and catering equipment"
query_vec = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_vec], n_results=3)

print(f"Query: '{query}'\n")
for i, doc in enumerate(results['documents'][0]):
    print(f"Match {i+1}: {doc}")
    print(f"Metadata: {results['metadatas'][0][i]}\n")

In [ ]:
query = "passenger refused safety instructions"
query_vec = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_vec], n_results=3)

print(f"Query: '{query}'\n")
for i, doc in enumerate(results['documents'][0]):
    print(f"Match {i+1}: {doc}")
    print(f"Metadata: {results['metadatas'][0][i]}\n")